In [31]:
import sys
sys.path.append('D:/gitRepository/my_medical_assistant-Qwen2.5-1.5B-Instruct')
from langchain_community.vectorstores import Chroma
from embedding.zhipuai_embedding import ZhipuAIEmbeddings


In [32]:
ZHIPUAI_API_KEY = "a7db401f88674a27a06a5d4698e0550d.Vd0YaLGBb0cRHegy"
embedding =ZhipuAIEmbeddings(zhipuai_api_key=ZHIPUAI_API_KEY)

In [33]:
persist_directory = 'D:/gitRepository/my_medical_assistant-Qwen2.5-1.5B-Instruct/new_vector_db'
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)
print(f"向量库中存储的数量：{vectordb._collection.count()}")

向量库中存储的数量：2822


In [34]:
question="发烧能吃小柴胡颗粒吗？吃多少合适？"
sim_docs = vectordb.similarity_search(question, k=20)
# print(sim_docs)
# print(f"检索到的内容数：{len(sim_docs)}")
for i, sim_doc in enumerate(sim_docs):
    print(f"检索到的第{i}个内容: \n{sim_doc.page_content}", end="\n--------------\n")

检索到的第0个内容: 
10
草热所致的各种症状，特别适用于缓解上述疾病的早期临床症状，如打喷嚏、流鼻涕、鼻
塞等症状。
【用法用量】口服，成人每12 小时服1 粒，24 小时内不应超过2 粒。
【禁忌】严重冠状动脉疾病、有精神病史者及严重高血压患者禁用。
【注意事项】
1. 本品一日剂量不得超过2 粒，疗程不超过3-7 天。症状未缓解者请咨询医师。
2. 对本品中任一组分过敏者禁用。
3. 驾驶机动车、船、操作机器以及高空作业者工作期间禁用。
4. 服用本品期间禁止饮酒。
5. 不能同时服用含有与本品成份相似的其他抗感冒药。
6. 心脏病、高血压、甲状腺疾病、糖尿病、前列腺肥大等患者使用本品前请咨询医师
或药师。
7. 孕妇及哺乳期妇女慎用。
8. 肝、肾功能不全者慎用。
--------------
检索到的第1个内容: 
35
【注意事项】
1.忌烟酒、辛辣、鱼腥食物。
2.不宜在服药期间同时服用温补性中药。
3.孕妇慎用。儿童应在医师指导下服用。
4.脾虚大便溏者慎用。
5.属风寒感冒咽痛者，症见恶寒发热、无汗、鼻流清涕者慎用。
6.扁桃体有化脓及全身高热者应去医院就诊。
7.服药3 天症状无缓解，应去医院就诊。
8.对本品过敏者禁用，过敏体质者慎用。
9.药品性状发生改变时禁止服用。
10.儿童必须在成人监护下使用。
11.请将此药品放在儿童不能接触的地方。
12.如正在服用其他药品，使用本品前请咨询医师或药师。
--------------
检索到的第2个内容: 
24
(2)胰升糖素治疗：胰升糖素皮下、肌肉或静脉注射，由于其作用时间短，且会再次出
现低血糖，因此在注射后仍要补充葡萄糖或进食，需继续观察以保证患者完全脱离危险期。
18. 对乙酰氨基酚栓
【主要成份】本品每粒含对乙酰氨基酚0.15 克。辅料为：焦亚硫酸钠、混合脂肪酸甘
油酯。
【性状】本品为乳白色至微黄色栓。
【简介/商品功效】用于普通感冒或流行性感冒引起的发热，也用于缓解轻至中度疼
痛如头痛、关节痛、偏头痛、牙痛、肌肉痛、神经痛、痛经。
【规格型号】0.15g*10 枚
【用法用量】直肠给药。6 岁以上儿童及成人一次1 粒，若持续发热或疼痛，可间隔
4-6 小时重复一次。24 小时内不超过4 粒。
【不良反应】偶见皮疹、荨麻疹、药热及粒细胞减少。长期大量用药会导致肝肾功能
异常。
【禁忌

In [ ]:
from FlagEmbedding import FlagReranker

# 1. 加载模型 (自动下载，利用 fp16 节省显存)
# use_fp16=True 非常重要，能省一半显存
reranker = FlagReranker('BAAI/bge-reranker-base', use_fp16=True) 

def rerank_results(query, retrieved_docs, top_k=3):
    """
    query: 用户的问题 (str)
    retrieved_docs: 向量检索回来的文档列表 (list of str)
    """
    if not retrieved_docs:
        return []
        
    # 1. 构造配对
    pairs = [[query, doc.page_content] for doc in retrieved_docs]
    
    # 2. 计算原始相关性得分 (Raw Logits)
    # BGE 的分数通常在 -10 (无关) 到 10 (极其相关) 之间
    original_scores = reranker.compute_score(pairs)
    
    # 3. --- 核心修改：应用加权逻辑 ---
    weighted_results = []
    
    for doc, score in zip(retrieved_docs, original_scores):
        final_score = score
        
        # --- 加权规则区域 (根据你的业务定制) ---
        
        # 规则 A: 优先推荐本院/自有药品库的数据 (通过 metadata 判断)
        # 假设你的 Excel 数据 metadata 里有 type="structured_drug"
        if doc.metadata.get("type") == "structured_drug":
            final_score += 3.0  # 强力加分 (相当于极大提升排名)
            # print(final_score)
            
        # 规则 B: 优先推荐包含“禁忌”或“警示”的关键安全信息
        if "禁忌" in doc.page_content or "慎用" in doc.page_content:
            final_score += 1.5  # 中等加分，安全第一
            
        # 规则 C: 降权非医疗来源的闲聊数据 (如果有)
        # if doc.metadata.get("category") == "chitchat":
        #     final_score -= 5.0  # 强力扣分
            
        # -----------------------------------
        
        weighted_results.append((doc, final_score))
    
    # 4. 按“最终分数”从高到低排序
    sorted_docs = sorted(weighted_results, key=lambda x: x[1], reverse=True)
    
    # 5. 返回前 Top_K 个文档内容 (如果你后续流程需要 metadata，这里也可以直接返回 doc 对象)
    return [doc[0] for doc in sorted_docs[:top_k]]

# --- 模拟流程 ---
user_query = "发烧能吃小柴胡颗粒吗？吃多少合适？"

# 1. 假设这是 ChromaDB 向量检索回来的 5 个粗排结果（可能有不相关的）
# vector_results = [
#     "阿司匹林适应症：用于缓解轻至中度疼痛如头痛...", # 相关但有风险
#     "小儿氨酚黄那敏颗粒：适用于缓解儿童普通感冒...",  # 相关
#     "布洛芬混悬液：用于儿童普通感冒或流行性感冒引起的发热...", # 最相关
#     "阿司匹林禁忌：12岁以下儿童禁用，可能引起瑞氏综合征...", # 极其重要！
#     "阿莫西林胶囊：本品为青霉素类抗生素..." # 完全不相关
# ]

# 2. Rerank 重排序
final_results = rerank_results(user_query, sim_docs, top_k=2)

print("重排序后的最佳结果：")
for res in final_results:
    print(res)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


6.251953125
4.2421875
重排序后的最佳结果：
page_content='【药品数据】
ID: 2593
名称: 仁和小柴胡颗粒
功能主治: 解表散热，疏肝和胃。用于外感病，邪犯少阳证，症见寒热往来、胸胁苦满、食欲不振、心烦喜呕、口苦咽于。
规格: 10g*9袋/盒
用法用量: 开水冲服。一次l袋（10克）一2袋（20克），一日3次。
注意事项: 1.忌烟、酒及辛辣、生冷、油腻食物。
2.不宜在服药期间同时服用滋补性中药。
3.风寒感冒者不适用。风寒表证者不宜使用。
4.糖尿病患者及有高血压、心脏病、肝病、肾病等慢性病严重者应在医师指导下服用。
5.儿童、孕妇、哺乳期妇女、年老体弱者应在医师指导下服用。
6.发热体温超过38.5度的患者，应去医院就诊。
7.服药3天症状无缓解，应去医院就诊。
8.对本品过敏者禁用，过敏体质者慎用。
9.本品性状发生改变时禁止使用。
10.儿童必须在成人监护下使用。
11.请将本品放在儿童不能接触的地方。
12.如正在使用其他药品，使用本品前请咨询医师或药师。
成分: 柴胡、姜半夏、黄芩、党参、甘草、生姜、大枣。辅料为：蔗糖。
性状: 本品为黄色至棕褐色的颗粒；味甜。
疗程: 疗程一般为1-3天
' metadata={'type': 'structured_drug', 'row_id': '2593', 'source': './knowledge_db\\药品说明书(最全).xlsx'}
page_content='【药品数据】
ID: 1775
名称: 瑞晴小柴胡颗粒(无蔗糖)
功能主治: 解表散热，疏肝和胃。用于外感病，邪犯少阳证，症见寒热往来、胸胁苦满、食欲不振、心烦喜呕、口苦咽干。
规格: 3g*20袋/盒
用法用量: 开水冲服，一次1-2袋，一日3次。
注意事项:  1.忌烟、酒及辛辣、生冷、油腻食物。 2.不宜在服药期间同时服用滋补性中成药。 3.高血压、心脏病、肝病、肾病等慢性病严重者应在医师指导下服用。 4.服药3天后或服药期间症状无改善，或症状加重，或出现新的严重症状如胸闷、心悸等应立即停药，并去医院就诊。 5.小儿、年老体弱者应在医师指导下服用。 6.孕妇慎用。 7.对本品过敏者禁用，过敏体质者慎用。 8.药品性状发生改变时禁止服用。 9.儿童必须在成人监护下使用。 10.请将此药品放在儿

In [ ]:
import jieba
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import Chroma

# --- 1. 准备阶段 (系统启动时加载) ---
# 假设 docs 是你所有文档的列表 (Document 对象)


# 初始化 Chroma (沿用你之前的代码)
# db = Chroma(...)

# --- 2. 混合检索函数 ---
def hybrid_search(query, top_k=20):
    # 注意：BM25 需要分词
    tokenized_corpus = [list(jieba.cut(doc.page_content)) for doc in sim_docs]
    bm25 = BM25Okapi(tokenized_corpus)
    # --- 路一：向量检索 (Dense) ---
    dense_results = vectordb.similarity_search(query, k=top_k)
    
    # --- 路二：BM25 检索 (Sparse) ---
    tokenized_query = list(jieba.cut(query))
    # BM25 返回的是文档内容，我们需要映射回 Document 对象
    # 这里为了演示简单，直接取 top_k
    bm25_docs = bm25.get_top_n(tokenized_query, sim_docs, n=top_k)
    
    # --- 3. 融合 (去重合并) ---
    # 使用字典推导式去重，Key 为文档内容或唯一ID
    combined_results = {doc.page_content: doc for doc in dense_results + bm25_docs}
    unique_docs = list(combined_results.values())
    
    return unique_docs

# --- 4. 接入 Reranker ---
# 假设你已经有了 reranker 模型
def final_retrieve(query):
    # 1. 混合召回 (可能返回 10~20 条)
    candidates = hybrid_search(query, top_k=20)
    
    # 2. Rerank 精排 (BGE-Reranker 会教做人)
    # 将 candidates 和 query 输入 Reranker，取 Top-2
    final_docs = rerank_results(query, candidates, top_k=2) 
    
    return final_docs

user_query = "腹部胀气能吃健胃消食片吗？吃多少合适？"
final_results = final_retrieve(user_query)

print("重排序后的最佳结果：")
for res in final_results:
    print(res)

4.609375
5.712890625
5.224609375
5.19140625
4.8427734375
5.47265625
4.2451171875
3.05975341796875
2.965576171875
0.611328125
5.357421875
-2.5859375
-1.7734375
重排序后的最佳结果：
page_content='【药品数据】
ID: 2589
名称: 健民健胃消食片
功能主治: 健胃消食。用于脾胃虚弱所致的食积，症见不思饮食、嗳腐酸臭、脘腹胀满；消化不良见上述证候者。
规格: 0.8g*32片/盒
用法用量: 口服，可以咀嚼，成人一次4-6片；儿童二岁至四岁一次2片，五岁至八岁一次3片，九岁至十四岁一次4片；一日3次。
注意事项: 1.饮食宜清淡，忌酒及辛辣、生冷、油腻食物。2.有高血压、心脏病、肝病、糖尿病、肾病等慢性病严重者应在医师指导下服用。3.儿童、孕妇、哺乳期妇女、年老体弱者应在医师指导下服用。4.服药3天症状无缓解，应去医院就诊。5.对本品过敏者禁用，过敏体质者慎用。6.本品性状发生改变时禁止使用。7.儿童必须在成人监护下使用。8.请将本品放在儿童不能接触的地方。9.如正在使用其他药品，使用本品前请咨询医师或药师。
成分: 太子参、陈皮、山药、麦芽（炒）、山楂。辅料为蔗糖、糊精、硬脂酸镁。
性状: 本品为浅棕黄色的薄膜衣异形片，除去包衣后显浅棕黄色；气略香，味微甜、酸。
疗程: 无
' metadata={'type': 'structured_drug', 'row_id': '2589', 'source': './knowledge_db\\药品说明书(最全).xlsx'}
page_content='【药品数据】
ID: 2380
名称: 泰华堂健胃消食片
功能主治: 健胃消食。用于脾胃虚弱所致的食积，症见不思饮食、嗳腐酸臭、脘腹胀满；消化不良见上述证候者。
规格: 0.5g*36片/盒
用法用量: 口服，可以咀嚼，成人一次4-6片；儿童二岁至四岁一次2片，五岁至八岁一次3片，九岁至十四岁一次4片；一日3次。
注意事项: 1.饮食宜清淡，忌酒及辛辣、生冷、油腻食物。2.有高血压、心脏病、肝病、糖尿病、肾病等慢性病严重者应在医师指导下服用。3.儿童、孕妇、哺乳期妇女、年老体弱者应在医师指导下服用